# Semantic Embeddings

**Notebook:** `03-semantic-embeddings.ipynb`

This notebook is part of the Personal Knowledge Engine project.

In [1]:
# Start coding here
# ==========================================================
# Notebook 03: Semantic Embeddings
# ==========================================================

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
chunks_df = pd.read_csv("data/processed/chunked_corpus.csv")

chunks_df.head()

,chunk_id,source,category,created_date,chunk_text
0,1,2026-06-14.md,journal,2026-06-14,# Daily Journal\n\nToday I read about local AI...
1,2,ai_resume_matcher.md,project,NaN,# AI Resume Matcher\n\nToday I worked on the F...
2,3,book.pdf,book,NaN,Praise for Natural Language Processing with Tr...
3,4,book.pdf,book,NaN,NLP than the creators of said library? Natura...
4,5,book.pdf,book,NaN,insight and deftly mixes research advances wit...


In [3]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Model loaded successfully.")

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model loaded successfully.


In [4]:
sample_chunk = chunks_df.iloc[0]["chunk_text"]

print(sample_chunk)

# Daily Journal

Today I read about local AI and privacy-first search systems.

Interesting ideas:
- Local embeddings
- Personal knowledge search
- Hybrid retrieval


In [5]:
embedding = model.encode(sample_chunk)

print("Embedding shape:", embedding.shape)

print(embedding[:10])

Embedding shape: (384,)
[-0.00545379 -0.03469334  0.01614941  0.04128626  0.04972453  0.03718477
  0.04241784  0.00098623 -0.01551806 -0.04396045]


In [6]:
chunk_texts = chunks_df["chunk_text"].tolist()

In [7]:
embeddings = model.encode(chunk_texts, show_progress_bar=True)

print(embeddings.shape)

Batches:   0%|          | 0/90 [00:00<?, ?it/s]

(2865, 384)


In [8]:
chunks_df["embedding"] = embeddings.tolist()

chunks_df.head()

,chunk_id,source,category,created_date,chunk_text,embedding
0,1,2026-06-14.md,journal,2026-06-14,# Daily Journal\n\nToday I read about local AI...,"[-0.005453762598335743, -0.03469334915280342, ..."
1,2,ai_resume_matcher.md,project,NaN,# AI Resume Matcher\n\nToday I worked on the F...,"[-0.1263018250465393, -0.06286957859992981, 0...."
2,3,book.pdf,book,NaN,Praise for Natural Language Processing with Tr...,"[-0.08763743191957474, -0.03917495906352997, 0..."
3,4,book.pdf,book,NaN,NLP than the creators of said library? Natura...,"[-0.09207145124673843, -0.03448568657040596, 0..."
4,5,book.pdf,book,NaN,insight and deftly mixes research advances wit...,"[-0.024674497544765472, -0.03824758157134056, ..."


In [9]:
text1 = "I am learning semantic search using FAISS."

text2 = "I am studying vector retrieval with Facebook AI Similarity Search."

In [10]:
embedding1 = model.encode(text1)

embedding2 = model.encode(text2)

In [11]:
score = cosine_similarity([embedding1], [embedding2])[0][0]

print(f"Cosine Similarity: {score:.4f}")

Cosine Similarity: 0.5360


In [12]:
def semantic_search(query, embeddings, dataframe, top_k=3):

    query_embedding = model.encode(query)

    scores = cosine_similarity([query_embedding], embeddings)[0]

    dataframe = dataframe.copy()

    dataframe["score"] = scores

    return dataframe.sort_values(by="score", ascending=False).head(top_k)

In [13]:
results = semantic_search(
    query="How does FAISS work?", embeddings=embeddings, dataframe=chunks_df, top_k=3
)

results[["chunk_id", "source", "score", "chunk_text"]]

,chunk_id,source,score,chunk_text
2033,2034,book.pdf,0.421314,he group (Figure 9-4).\nFigure 9-4. The struct...
2037,2038,book.pdf,0.367873,"function, where we need to do the least compa..."
2025,2026,book.pdf,0.363915,a transformer on the limited data we\nhave.\nE...


In [15]:
import pickle

with open("data/embeddings/chunk_embeddings.pkl", "wb") as file:

    pickle.dump(embeddings, file)

print("Embeddings saved successfully.")

Embeddings saved successfully.


In [16]:
with open("data/embeddings/chunk_embeddings.pkl", "rb") as file:

    loaded_embeddings = pickle.load(file)

print(loaded_embeddings.shape)

(2865, 384)
